# Complete real 30-epoch training for sizes 70, 140, 280 — paired CE and Poly-1

**Follow-up batch in the independent full-epoch study.** The first two 420-image runs (CE and Poly-1, seed 20261028) are already completed and independently checked. This notebook executes **six additional runs**: sizes 70, 140 and 280, each with Cross-Entropy and Poly-1, all with seed **20261028**, 30 **true full dataset epochs**, ImageNet-pretrained ResNet18 at 448 × 448, AdamW, batch 4, in **Colab T4**.

The **same fixed 180-image validation pool** as the verified 420-image experiments is enforced before training every size. It was selected exclusively from official TRAIN and excludes the entire frozen 420-image pool. The previously inspected official TEST dataset is never opened for model selection.

**Important:** Keep the older 216-step study unchanged. All new run files are backed up to `MyDrive/Nazih_Coursework/06_Full_Epoch_Pilot/n<size>_seed20261028_<loss>/`. The notebook *skips verified completed runs* and *resumes interrupted runs* at full-epoch boundaries. Do not launch two copies concurrently. Each run can be stopped and later restarted, but simultaneous runs would conflict on shared Drive files.

Before running: **Runtime → Change runtime type → T4 GPU**; then **Runtime → Run all** and authorize Drive. The machine must have access to the existing Drive workspace and official ZIP.


In [ ]:
import torch
import torchvision
assert torch.cuda.is_available(), "GPU unavailable: Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "| torchvision:", torchvision.__version__)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import subprocess, sys, os, shutil, hashlib, json
REPOSITORY = "https://github.com/ID9999999999999/DL-Computer-vision.git"
BRANCH = "nazih-training-set-size-20261109"
REPO = Path("/content/nazih_full_epoch_project")
PROJECT = REPO / "COURSEWORK_NAZIH"
DRIVE_BASE = Path("/content/drive/MyDrive/Nazih_Coursework")
DRIVE_OUT = DRIVE_BASE / "06_Full_Epoch_Pilot"
DRIVE_DATASET = DRIVE_BASE / "05_Official_Dataset" / "KolektorSDD2.zip"

assert DRIVE_BASE.is_dir(), f"Existing project Drive folder not visible: {DRIVE_BASE}"
assert DRIVE_OUT.is_dir(), f"Expected backup folder missing (created earlier): {DRIVE_OUT}"
assert DRIVE_DATASET.is_file(), f"Official ZIP missing on Drive: {DRIVE_DATASET}"

if not (REPO / ".git").is_dir():
    assert not REPO.exists(), "Refusing to overwrite unrelated /content project"
    subprocess.run(["git", "clone", "--single-branch", "--branch", BRANCH,
                    "--depth", "1", REPOSITORY, str(REPO)], check=True)
else:
    status = subprocess.check_output(
        ["git", "-C", str(REPO), "status", "--porcelain"], text=True
    ).strip()
    assert not status, "Unexpected local changes in Colab clone"
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH], check=True)

assert (PROJECT / "scripts" / "full_epoch_pilot.py").is_file()
assert (PROJECT / "protocol.json").is_file()
output_link = PROJECT / "data" / "full_epoch_pilot"
output_link.parent.mkdir(parents=True, exist_ok=True)
if output_link.is_symlink():
    assert output_link.resolve() == DRIVE_OUT.resolve(), "Output points to the wrong Drive folder!"
elif output_link.exists():
    raise RuntimeError("Refusing to overwrite existing non-symlink output directory")
else:
    output_link.symlink_to(DRIVE_OUT, target_is_directory=True)

print("✅ Code cloned and persistent Drive output linked")
print("✅ Existing historical results remain untouched")
print("Output:", output_link.resolve())

In [ ]:
EXPECTED_SHA256 = "edcdb486809b24f1d17b785e30c52fafc5999554dd5fe18ddf77b61ceb6f36a8"
LOCAL_DATASET = Path("/content/KolektorSDD2.zip")
def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for piece in iter(lambda: f.read(4 * 1024 * 1024), b""):
            h.update(piece)
    return h.hexdigest()

if not LOCAL_DATASET.exists():
    free = shutil.disk_usage("/content").free
    assert free > DRIVE_DATASET.stat().st_size + 1024**3, "Not enough Colab disk space"
    shutil.copy2(DRIVE_DATASET, LOCAL_DATASET)

assert sha256(LOCAL_DATASET) == EXPECTED_SHA256, "Official dataset SHA256 differs. Stop."
print("✅ Dataset copied from Google Drive and SHA256 verified")
print("ZIP bytes:", LOCAL_DATASET.stat().st_size)

In [ ]:
SCRIPT = PROJECT / "scripts" / "full_epoch_pilot.py"
RUNS = [(size, loss) for size in (70, 140, 280) for loss in ("ce", "poly1")]
SEED = 20261028
TOTAL_EPOCHS = 30
REFERENCE_SPLIT = DRIVE_OUT / "n420_seed20261028_ce" / "split.json"
assert REFERENCE_SPLIT.is_file(), "Verified 420 CE split is required, but is missing"
assert sha256(REFERENCE_SPLIT) == "1be0f86837238f81495121ae485b31d65ddf2dce512f20c540734057e516bcb0", (
    "Frozen validation reference changed; stop"
)
reference = json.loads(REFERENCE_SPLIT.read_text(encoding="utf-8"))
reference_validation = reference["validation_rows"]
assert len(reference_validation) == 180
assert sum(r["label"] == 0 for r in reference_validation) == 150
assert sum(r["label"] == 1 for r in reference_validation) == 30

import sys as _sys
_sys.path.insert(0, str(SCRIPT.parent))
from full_epoch_pilot import fixed_splits
for n in (70, 140, 280):
    training, validation, frozen_sha = fixed_splits(LOCAL_DATASET, n)
    assert validation == reference_validation, "Validation differs from the 420 CE reference!"
    assert not {x["sha256"] for x in training} & {x["sha256"] for x in validation}
    assert frozen_sha == reference["manifest_sha256"]
    print(f"✅ Size {n}: {len(training)} training images; common 180-image validation identical")

print("✅ SIX PAIRED EXPERIMENTS PREFLIGHT VERIFIED — NO TEST IMAGES OPENED")

## Execute six actual full-epoch runs (resumable)

Each run is kept in its own Drive-backed directory. Already-completed runs are SHA256-verified and skipped. Only missing or interrupted runs are trained. Colab can disconnect; restart this notebook and run all again to resume safely. The original 420-image experiments are never reopened or overwritten.


In [ ]:
completed = []
for size, loss in RUNS:
    name = f"n{size}_seed{SEED}_{loss}"
    run_dir = DRIVE_OUT / name
    done = run_dir / "COMPLETE.json"
    args = [
        "--archive", str(LOCAL_DATASET),
        "--size", str(size), "--seed", str(SEED),
        "--loss", loss, "--epochs", str(TOTAL_EPOCHS),
        "--batch-size", "4", "--device", "cuda"
    ]
    if done.exists():
        print("PRESERVED completed run:", name)
    else:
        print("START/RESUME run:", name, flush=True)
        subprocess.run([_sys.executable, str(SCRIPT), *args], cwd=PROJECT, check=True)
    assert done.is_file(), f"Run not complete: {name}"
    record = json.loads(done.read_text(encoding="utf-8"))
    hist = record["history"]
    assert record["official_test_evaluated"] is False
    assert record["identity"]["loss"] == loss and record["identity"]["size"] == size
    assert record["identity"]["seed"] == SEED
    assert len(hist) == TOTAL_EPOCHS
    assert [x["full_epoch"] for x in hist] == list(range(1, TOTAL_EPOCHS + 1))
    steps = (size + 3) // 4
    assert all(h["training_optimizer_steps"] == steps for h in hist)
    assert record["identity"]["total_planned_optimizer_steps"] == TOTAL_EPOCHS * steps
    assert sha256(run_dir / "best_model.pt") == record["best_model_sha256"]
    split = json.loads((run_dir / "split.json").read_text(encoding="utf-8"))
    assert split["validation_rows"] == reference_validation
    completed.append({"size": size, "loss": loss,
                      "best_epoch": record["best_epoch"],
                      "macro_f1": record["best_validation_macro_f1"],
                      "steps": TOTAL_EPOCHS * steps})
    print(f"✅ VERIFIED {name}: best={record['best_epoch']} Macro-F1={record['best_validation_macro_f1']:.4f}")

assert len(completed) == 6
print("\n✅ ALL SIX NEW 30-FULL-EPOCH TRAININGS COMPLETED")
print("✅ Previous 24 historical evaluations and the paired 420 experiments preserved")

In [ ]:
from pprint import pprint
assert len(completed) == 6, "Complete the previous cell first"
pprint(completed)

# Compact summary, no overwrite of model/checkpoint histories.
summary_path = DRIVE_OUT / "BATCH_SEED20261028_SIZES70_140_280.json"
summary = {
    "status": "six_full_epoch_runs_verified",
    "epochs_each": TOTAL_EPOCHS,
    "validation_policy": "same verified 180-image train-only holdout across all sizes",
    "official_test_evaluated": False,
    "runs": completed
}
if summary_path.exists():
    assert json.loads(summary_path.read_text(encoding="utf-8")) == summary, (
        "Existing summary differs; refusing overwrite"
    )
else:
    summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
print("✅ SAVED:", summary_path)
print("✅ Completed full-epoch study count: 8/24, if both verified 420-image runs retained.")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
for loss in ("ce", "poly1"):
    subset = sorted((r for r in completed if r["loss"] == loss), key=lambda r: r["size"])
    ax.plot([x["size"] for x in subset],
            [100 * x["macro_f1"] for x in subset], marker="o", label=loss.upper())
ax.set_title("Internal train-only validation after 30 full epochs (seed 20261028)")
ax.set_xlabel("Number of training images")
ax.set_ylabel("Best validation Macro-F1 (%)")
ax.grid(alpha=0.25)
ax.legend()
plt.show()
print("Exploratory validation only — no fresh independent test estimates.")